In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentencepiece transformers datasets accelerate nlpaug nltk

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 8.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
from sklearn.metrics import classification_report

set_seed(42)

train_df = pd.read_csv('train.csv')
val_df = pd.read_csv('val.csv')
test_df = pd.read_csv('test.csv')

label_map = {
    "similar_items": 0,
    "graph_pairing": 1,
    "color_variant": 2,
    "chit_chat": 3,
    "composite_intent": 4
}

for df in [train_df, val_df, test_df]:
    df['intent'] = df['intent'].str.strip()
    df['labels'] = df['intent'].map(label_map)
    df.dropna(subset=['labels'], inplace=True)
    df['labels'] = df['labels'].astype(int)

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)
print('data loaded')

data loaded


In [ ]:
MODEL_ID = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_function(examples):
    return tokenizer(examples["query"], padding="max_length", truncation=True, max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True).remove_columns(["query", "intent"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True).remove_columns(["query", "intent"])
tokenized_test = test_dataset.map(tokenize_function, batched=True).remove_columns(["query", "intent"])

print('tokenization completed')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3574 [00:00<?, ? examples/s]

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

tokenization completed


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=5,
    id2label={v: k for k, v in label_map.items()},
    label2id=label_map
)

training_args = TrainingArguments(
    output_dir="./results_5_classes",
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_steps=125,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    target_names = ["similar_items", "graph_pairing", "color_variant", "chit_chat", "composite_intent"]
    report = classification_report(labels, predictions, target_names=target_names, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "f1_macro": report["macro avg"]["f1-score"]
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('training started')
trainer.train()
trainer.save_model("/content/drive/MyDrive/model_cache/intent_classifier_roberta")
print('model saved')

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


training started


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.171729,0.953020,0.952777
2,No log,0.247426,0.946309,0.943045
3,0.436320,0.096883,0.977629,0.978459
4,0.436320,0.109670,0.979866,0.980624
5,0.064563,0.102735,0.984340,0.985378


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved


In [ ]:
print('evaluation started')
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
print(classification_report(labels, preds, target_names=["similar_items", "graph_pairing", "color_variant", "chit_chat", "composite_intent"], digits=4))

evaluation started


                  precision    recall  f1-score   support

   similar_items     0.9320    0.9897    0.9600        97
   graph_pairing     1.0000    1.0000    1.0000        98
   color_variant     0.9684    0.9892    0.9787        93
       chit_chat     1.0000    0.9524    0.9756        63
composite_intent     0.9890    0.9375    0.9626        96

        accuracy                         0.9754       447
       macro avg     0.9779    0.9738    0.9754       447
    weighted avg     0.9763    0.9754    0.9754       447



In [ ]:
import torch
import torch.nn.functional as F

edge_cases = [
    "cheaper alternative to this jacket",
    "what shoes go with this dress for a wedding",
    "do you have this exact same shirt in dark blue",
    "hello where is my refund",
    "find me a dupe for this bag but in black",
    "need this blazer in navy and what pants to match",
    "similar pantsuit in gray and shoes for it",
    "skrrt banana cloud microwave",
    "red leather boots with silver buckles",
    "date night outfit layer"
]

model.eval()
print('inference started')

for q in edge_cases:
    inputs = tokenizer(q, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=-1)
    pred_idx = torch.argmax(probs, dim=-1).item()

    label = model.config.id2label[pred_idx]
    score = probs[0][pred_idx].item()

    print(f"Q: '{q}' | Pred: {label} | Score: {score:.4f}")

print('inference completed')

inference started
Q: 'cheaper alternative to this jacket' | Pred: similar_items | Score: 0.9993
Q: 'what shoes go with this dress for a wedding' | Pred: graph_pairing | Score: 0.9993
Q: 'do you have this exact same shirt in dark blue' | Pred: color_variant | Score: 0.9993
Q: 'hello where is my refund' | Pred: chit_chat | Score: 0.9991
Q: 'find me a dupe for this bag but in black' | Pred: composite_intent | Score: 0.9994
Q: 'need this blazer in navy and what pants to match' | Pred: composite_intent | Score: 0.9994
Q: 'similar pantsuit in gray and shoes for it' | Pred: composite_intent | Score: 0.9994
Q: 'skrrt banana cloud microwave' | Pred: chit_chat | Score: 0.9990
Q: 'red leather boots with silver buckles' | Pred: composite_intent | Score: 0.9949
Q: 'date night outfit layer' | Pred: similar_items | Score: 0.9931
inference completed
